# Tool-Router Ladder: Stage F serving benchmarks (F2-F6)

Everything replays the recorded forced-eval prompts, so **no API quota**. Three parts; set `PART` below.

| PART | What | Where | Estimate |
|---|---|---|---|
| `"smoke"` | every path once at toy size (0.5B): guided decoding, a GPTQ checkpoint, a 2-adapter server | 1 T4 | ~15 min |
| `"accuracy"` | `generate.py` over all 1,416 steps: **F4** GPTQ x 3 sizes x 3 catalogs; **F3** 3B guided x 3 catalogs (with log-probs) + 3B unguided c80 log-probs (the calibration baseline) | **both T4s**, two runs at a time | ~7 h wall (~14 run-hours) |
| `"latency"` | `serve_bench.py` sweeps, concurrency 1/8/32 x 3 repeats x 48 requests: **F6** 3B SFT at c16/40/80, **F5** the same with prefix caching off, **F2** 3 adapters on one server (single vs mixed), **F3** guided at c16/c80, **F4** 3B GPTQ and 1.5B fp16 (memory-matched) at c16 | GPU 0 only, one run at a time | ~9 h |

**Order:** run `kaggle_quantize.ipynb` first (F4 needs the `-sft-gptq` repos), then `smoke`, then
`accuracy` and `latency` (separate sessions; either order). Each run is pushed to the Hub when it finishes
and skipped next time, so a session that times out just gets rerun.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, the tool-router dataset attached (it holds
`prompts-c16/40/80.jsonl`), secret `HF_TOKEN` ticked. **Run:** Save Version -> **Save & Run All (Commit)**.

**Back on the Mac:** `make pull` (completions) and `make pull ARGS="--free --pattern 'serve-bench-*'"` (latency
reports), then the scoring commands printed by the last cell.

In [ ]:
PART = "smoke"   # "smoke" -> "accuracy" / "latency"

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
# Pinned: every forced-eval row so far is vLLM 0.30.0 (with a CUDA 13 torch).
# Kaggle's image now ships a CUDA 12.8 torchaudio, and transformers imports torchaudio
# whenever it is installed, so it crashes against that torch. Nothing here uses audio.
!pip install -q vllm==0.30.0
!pip uninstall -y -q torchaudio

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os, shutil, subprocess, threading, queue
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
hits = [h for h in hits if os.path.exists(os.path.join(os.path.dirname(h), "prompts-c16.jsonl"))]
assert hits, "attach the tool-router dataset (Add Input): no prompts-c16.jsonl next to a manifest.json"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"
OUT = "/kaggle/working/out"
os.makedirs(OUT, exist_ok=True)
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

# Resume across sessions: bring back what an earlier session pushed.
from huggingface_hub import HfApi, snapshot_download
REPO = f"{HF_USER}/tool-router-results"
api = HfApi()
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
snap = snapshot_download(REPO, repo_type="dataset", allow_patterns=[
    "completions/completions-*-gptq-c*", "completions/completions-*-guided-c*",
    "completions/completions-qwen3b-sft-c80-logprobs*", "results/serve-bench-*"])
for f in glob.glob(f"{snap}/completions/*") + glob.glob(f"{snap}/results/serve-bench-*"):
    shutil.copy(f, OUT)
print("already done:", sorted(os.listdir(OUT)))

def push(path, folder):
    for f in (path, path.replace(".jsonl", ".meta.json")):
        if os.path.exists(f):
            api.upload_file(path_or_fileobj=f, path_in_repo=f"{folder}/{os.path.basename(f)}",
                            repo_id=REPO, repo_type="dataset")

def whole(path):  # a finished completions file has all 1,416 steps
    return os.path.exists(path) and sum(1 for _ in open(path)) == 1416

def run(cmd, gpu, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    with open(log, "w") as fh:
        rc = subprocess.run(cmd, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print(open(log).read()[-4000:])
    return rc

def gen(size, catalog, out, *extra):
    return ["python", "src/generate.py", "--config", f"configs/{size}.yaml",
            "--prompts", f"{DATA}/prompts-c{catalog}.jsonl", "--out", out, *extra]

def bench(size, catalog, label, *extra):
    return ["python", "src/serve_bench.py", "--config", f"configs/{size}.yaml", "--catalog", str(catalog),
            "--prompts", f"{DATA}/prompts-c{catalog}.jsonl", "--label", label,
            "--out", f"{OUT}/serve-bench-{label}.json", *extra]

def gptq(size):
    return f"{HF_USER}/tool-router-{size}-sft-gptq"

def sft(size, name="sft"):
    return f"{name}={HF_USER}/tool-router-{size}-{name}"

In [ ]:
### PART "smoke": every new path once, 0.5B, a few prompts (~15 min). Nothing is pushed.
if PART == "smoke":
    S = "/kaggle/working/smoke"; os.makedirs(S, exist_ok=True)
    checks = {
        "guided generate (F3)": gen("qwen05b", 16, f"{S}/guided.jsonl", "--adapter", f"{HF_USER}/tool-router-qwen05b-sft",
                                    "--guided-catalog", "16", "--logprobs", "--smoke", "--limit", "6"),
        "GPTQ generate (F4)": gen("qwen05b", 16, f"{S}/gptq.jsonl", "--model", gptq("qwen05b"), "--smoke", "--limit", "6"),
        "2-adapter mixed server, guided (F2, F3)": bench("qwen05b", 16, "smoke-mix", "--adapter", sft("qwen05b"),
                                    "--adapter", sft("qwen05b", "grpo"), "--mix", "--guided", "--smoke",
                                    "--levels", "1", "4", "--per-level", "4", "--warmup", "1", "--repeats", "1"),
        "GPTQ server, prefix caching off (F4, F5)": bench("qwen05b", 16, "smoke-gptq", "--model", gptq("qwen05b"),
                                    "--no-prefix-caching", "--smoke", "--levels", "1", "--per-level", "4",
                                    "--warmup", "1", "--repeats", "1"),
    }
    for name, cmd in checks.items():
        rc = run(cmd, 0, f"{S}/{name.split()[0]}.log")
        print(f"{'OK ' if rc == 0 else 'FAIL'} {name}")
        assert rc == 0, f"smoke failed: {name} (log above)"
    # Eyeball: guided completions must be a reply or <tool_call> blocks with catalog names.
    for line in open(f"{S}/guided.jsonl"):
        print(repr(__import__("json").loads(line)["completion"][:300]))
    print(open(f"{OUT}/serve-bench-smoke-mix.json").read()[:1500])

In [ ]:
### PART "accuracy": F4 GPTQ (3 sizes x 3 catalogs) and F3 guided (3B), two runs at a time on the two T4s.
if PART == "accuracy":
    jobs = queue.Queue()
    for size in ["qwen3b", "qwen15b", "qwen05b"]:          # longest first, so the tail is short
        for c in (80, 40, 16):
            jobs.put((f"{size}-sft-gptq-c{c}", gen(size, c, f"{OUT}/completions-{size}-sft-gptq-c{c}.jsonl",
                                                   "--model", gptq(size))))
    for c in (80, 40, 16):
        jobs.put((f"qwen3b-sft-guided-c{c}", gen("qwen3b", c, f"{OUT}/completions-qwen3b-sft-guided-c{c}.jsonl",
                                                 "--adapter", f"{HF_USER}/tool-router-qwen3b-sft",
                                                 "--guided-catalog", str(c), "--logprobs")))
    # Calibration baseline for F3: the same 3B SFT, unguided, with log-probs, on vLLM.
    jobs.put(("qwen3b-sft-c80-logprobs", gen("qwen3b", 80, f"{OUT}/completions-qwen3b-sft-c80-logprobs.jsonl",
                                             "--adapter", f"{HF_USER}/tool-router-qwen3b-sft", "--logprobs")))
    failed = []

    def worker(gpu):
        while True:
            try:
                name, cmd = jobs.get_nowait()
            except queue.Empty:
                return
            out = cmd[cmd.index("--out") + 1]
            if whole(out):
                print(f"[gpu {gpu}] {name}: done earlier, skipped"); continue
            print(f"[gpu {gpu}] {name}: start")
            rc = run(cmd, gpu, f"{OUT}/{name}.log")      # generate.py resumes a partial file
            if rc == 0 and whole(out):
                push(out, "completions"); print(f"[gpu {gpu}] {name}: pushed")
            else:
                failed.append(name); print(f"[gpu {gpu}] {name}: FAILED (rc {rc})")

    threads = [threading.Thread(target=worker, args=(g,)) for g in (0, 1)]
    for t in threads: t.start()
    for t in threads: t.join()
    assert not failed, f"failed: {failed} (logs printed above); rerun to retry them"

In [ ]:
### PART "latency": serve_bench sweeps, one at a time on GPU 0 (nothing else on the machine).
if PART == "latency":
    A3 = ["--adapter", sft("qwen3b")]
    runs = []
    for c in (16, 40, 80):
        runs += [(f"f6-qwen3b-sft-c{c}", bench("qwen3b", c, f"f6-qwen3b-sft-c{c}", *A3)),
                 (f"f5-qwen3b-sft-noprefix-c{c}", bench("qwen3b", c, f"f5-qwen3b-sft-noprefix-c{c}", *A3,
                                                        "--no-prefix-caching"))]
    three = ["--adapter", sft("qwen3b"), "--adapter", sft("qwen3b", "grpo"), "--adapter", sft("qwen3b", "dpo2")]
    runs += [
        ("f2-qwen3b-3lora-single-c16", bench("qwen3b", 16, "f2-qwen3b-3lora-single-c16", *three)),
        ("f2-qwen3b-3lora-mix-c16", bench("qwen3b", 16, "f2-qwen3b-3lora-mix-c16", *three, "--mix")),
        ("f3-qwen3b-sft-guided-c16", bench("qwen3b", 16, "f3-qwen3b-sft-guided-c16", *A3, "--guided")),
        ("f3-qwen3b-sft-guided-c80", bench("qwen3b", 80, "f3-qwen3b-sft-guided-c80", *A3, "--guided")),
        ("f4-qwen3b-sft-gptq-c16", bench("qwen3b", 16, "f4-qwen3b-sft-gptq-c16", "--model", gptq("qwen3b"))),
        ("f4-qwen15b-sft-c16", bench("qwen15b", 16, "f4-qwen15b-sft-c16", "--adapter", sft("qwen15b"))),
    ]
    failed = []
    for name, cmd in runs:
        out = f"{OUT}/serve-bench-{name}.json"
        if os.path.exists(out):
            print(f"{name}: done earlier, skipped"); continue
        print(f"===== {name} =====", flush=True)
        if run(cmd, 0, f"{OUT}/{name}.log") == 0:
            print(open(f"{OUT}/{name}.log").read()[-1200:]); push(out, "results")
        else:
            failed.append(name)
    assert not failed, f"failed: {failed} (logs printed above); rerun to retry them"

In [ ]:
### What to run on the Mac afterwards.
print("""make pull ARGS="--pattern '*-gptq-c*'"
make pull ARGS="--pattern '*-guided-c*'"
make pull ARGS="--pattern '*qwen3b-sft-c80-logprobs*'"
make pull ARGS="--free --pattern 'serve-bench-*'"
make score-f""")